In [3]:
import numpy as np
import torch
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_curve


In [4]:
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [5]:
X = np.load("../outputs/xvectors/xvectors.npy")      # (N, 192)
utterances = np.load("../outputs/xvectors/utterances.npy")  # list of paths

print("X shape:", X.shape)
print("Utterances:", len(utterances))


X shape: (4874, 512)
Utterances: 4874


In [6]:
speakers = [u.split("/")[0] for u in utterances]

spk2id = {s: i for i, s in enumerate(sorted(set(speakers)))}
y = torch.tensor([spk2id[s] for s in speakers], dtype=torch.long)
y = y.to("cpu")
print("Number of speakers:", len(spk2id))


Number of speakers: 40


In [7]:
protocol = "../data/voxceleb1/test/veri_test2.txt"

trials = []
labels = []

with open(protocol) as f:
    for line in f:
        lab, u1, u2 = line.strip().split()
        trials.append((u1, u2))
        labels.append(int(lab))

labels = np.array(labels)
print("Trials:", len(trials))
utt2idx = {u: i for i, u in enumerate(utterances)}
X = torch.tensor(X, dtype=torch.float)
device = "cpu"
X = X.to(device)

Trials: 37611


In [8]:
def compute_eer_and_min_dcf(scores, labels):
   p_target=0.01
   c_miss=1.0
   c_fa=1.0

   fpr, tpr , _ = roc_curve(labels, scores, pos_label=1)

   fnr = 1 - tpr 

   dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
   min_dcf = np.min(dcf)

   eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
   return eer, min_dcf

In [15]:
sim_matrix = cosine_similarity(X)

def build_threshold_graph(knn_value):
    K = knn_value
    N = sim_matrix.shape[0]
    edge_index = []

    for i in range(sim_matrix.shape[0]):
        knn = np.argsort(sim_matrix[i])[::-1][1:K+1]
        for j in knn:
          edge_index.append([i, j])

    edge_index = torch.tensor(edge_index, dtype=torch.long).T
    class GCN(torch.nn.Module):
        def __init__(self, in_dim=512, hidden=128, out_dim=512):
            super().__init__()
            self.conv1 = GCNConv(in_dim, hidden)
            self.conv2 = GCNConv(hidden, out_dim)
        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index)
            x = F.relu(x)
            x = self.conv2(x, edge_index)
            return x
    device = "cpu"

    model = GCN().to(device)
    edge_index = edge_index.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    epochs = 15

    for epoch in range(epochs):
     model.train()
     optimizer.zero_grad()

     Z = model(X, edge_index)
     Z = F.normalize(Z, dim=1)

     sim = torch.matmul(Z, Z.T)  # (N, N)
     label_eq = (y.unsqueeze(1) == y.unsqueeze(0)).float()

     loss = - (label_eq * sim).sum() / label_eq.sum()

     loss.backward()
     optimizer.step()
    scores = np.zeros(len(trials), dtype=np.float32)

    for k, (u1, u2) in enumerate(trials):
      i = utt2idx[u1]
      j = utt2idx[u2]
      scores[k] = np.dot(
      Z[i].detach().cpu().numpy(),
      Z[j].detach().cpu().numpy()
      )
    eer, min_dcf = compute_eer_and_min_dcf(scores, labels) # type: ignore

    return eer, min_dcf





/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [11]:
def run_k_times(K, runs=5):
    eers, dcfs = [], []
    for _ in range(runs):
        eer, dcf = build_threshold_graph(K)
        eers.append(eer)
        dcfs.append(dcf)
    return np.mean(eers), np.mean(dcfs)


In [17]:
knn_value = [40]
results_err = {}
results_dcf = {}

for t in knn_value:
    err, min_dcf = run_k_times(t)
    print(f"GCN-refined EER for KNN {t}: {err * 100:.2f}%")
    print(f"GCN-refined MinDCF for KNN {t}: {min_dcf:.5f}")


GCN-refined EER for KNN 40: 4.62%
GCN-refined MinDCF for KNN 40: 0.00336
